# VectorDB 과제 — 내풀이 v1

## 공통 설치 라이브러리

In [ ]:
!pip install sentence-transformers openai chromadb rank-bm25 scikit-learn httpx rouge-score -q

In [ ]:
import os
import numpy as np
import time
import chromadb
from dotenv import load_dotenv
load_dotenv()

---
## 문제 1. 멀티 모델 임베딩 비교 벤치마크 구현 (10점)

In [ ]:
from sentence_transformers import SentenceTransformer
from openai import OpenAI
import numpy as np
import time

EVAL_PAIRS = [
    ("인공지능이 의료 분야를 혁신하고 있다.", "AI 기술이 병원에서 활용되고 있다.", True),
    ("오늘 서울 날씨가 맑습니다.", "서울의 하늘이 화창하고 좋다.", True),
    ("주식 시장이 크게 하락했다.", "금융 위기가 발생했다.", True),
    ("인공지능이 의료 분야를 혁신하고 있다.", "오늘 저녁에 파스타를 만들 예정이다.", False),
    ("서울에서 부산까지 KTX로 2시간 걸린다.", "고양이가 소파에서 낮잠을 자고 있다.", False),
    ("딥러닝 모델의 학습률을 조정했다.", "맛있는 김치찌개 레시피를 알려주세요.", False),
    ("클라우드 서버 비용이 증가하고 있다.", "AWS 요금이 올랐다.", True),
    ("자연어 처리 기술이 발전하고 있다.", "NLP 분야에서 새로운 모델이 등장했다.", True),
    ("오늘 점심에 비빔밥을 먹었다.", "내일 회의가 오후 3시에 있다.", False),
    ("파이썬은 데이터 분석에 널리 사용된다.", "Python은 ML/AI에서 인기있는 언어이다.", True),
]

class EmbeddingBenchmark:
    def __init__(self):
        self.local_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
        self.openai_client = OpenAI()
    
    def embed_local(self, texts):
        return self.local_model.encode(texts)
    
    def embed_api(self, texts, model="text-embedding-3-small"):
        response = self.openai_client.embeddings.create(input=texts, model=model)
        return np.array([d.embedding for d in response.data])
    
    def cosine_similarity(self, a, b):
        return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
    
    def evaluate(self, embed_func, threshold=0.5):
        start = time.time()
        
        sentences_a = [p[0] for p in EVAL_PAIRS]
        sentences_b = [p[1] for p in EVAL_PAIRS]
        labels = [p[2] for p in EVAL_PAIRS]
        
        emb_a = embed_func(sentences_a)
        emb_b = embed_func(sentences_b)
        
        similarities = [self.cosine_similarity(emb_a[i], emb_b[i]) for i in range(len(EVAL_PAIRS))]
        predictions = [sim >= threshold for sim in similarities]
        
        tp = sum(1 for p, l in zip(predictions, labels) if p and l)
        fp = sum(1 for p, l in zip(predictions, labels) if p and not l)
        fn = sum(1 for p, l in zip(predictions, labels) if not p and l)
        tn = sum(1 for p, l in zip(predictions, labels) if not p and not l)
        
        accuracy = (tp + tn) / len(labels)
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        
        pos_sims = [s for s, l in zip(similarities, labels) if l]
        neg_sims = [s for s, l in zip(similarities, labels) if not l]
        
        elapsed = (time.time() - start) * 1000
        
        return {
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "avg_sim_positive": np.mean(pos_sims),
            "avg_sim_negative": np.mean(neg_sims),
            "elapsed_ms": elapsed
        }

bench = EmbeddingBenchmark()
local_result = bench.evaluate(bench.embed_local)
api_result = bench.evaluate(bench.embed_api)

print("=" * 65)
print(f"{'지표':>20} | {'오픈소스 (MiniLM)':>16} | {'API (OpenAI)':>16}")
print("-" * 65)
for key in ["accuracy", "precision", "recall", "avg_sim_positive", "avg_sim_negative", "elapsed_ms"]:
    lv = local_result[key]
    av = api_result[key]
    fmt = ".1f" if key == "elapsed_ms" else ".4f"
    print(f"{key:>20} | {lv:>16{fmt}} | {av:>16{fmt}}")

---
## 문제 2. 임베딩 차원 축소 + 시각화 분석 (10점)

In [ ]:
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

DOCUMENTS = [
    ("딥러닝 모델의 학습률을 조정하는 방법", "tech"),
    ("자연어 처리에서 트랜스포머 아키텍처의 역할", "tech"),
    ("GPU를 활용한 대규모 모델 학습 최적화", "tech"),
    ("강화학습 에이전트의 보상 함수 설계", "tech"),
    ("김치찌개를 맛있게 끌이는 비법", "food"),
    ("이탈리안 파스타의 정통 레시피", "food"),
    ("건강한 샐러드 드레싱 만들기", "food"),
    ("한국 전통 떡 만드는 방법", "food"),
    ("제주도 3박 4일 여행 코스 추천", "travel"),
    ("유럽 배낙여행 필수 준비물 리스트", "travel"),
    ("도쿄 맛집 투어 가이드", "travel"),
    ("발리 서핑 스팟 베스트 5", "travel"),
]

texts = [d[0] for d in DOCUMENTS]
labels = [d[1] for d in DOCUMENTS]

model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
embeddings = model.encode(texts)

pca = PCA(n_components=2)
reduced = pca.fit_transform(embeddings)

sil_score = silhouette_score(embeddings, labels)

print(f"실루엣 스코어: {sil_score:.4f}")
print(f"(1에 가까울수록 클러스터가 잘 분리됨)\n")

centroids = {}
for label in set(labels):
    indices = [i for i, l in enumerate(labels) if l == label]
    centroids[label] = np.mean(reduced[indices], axis=0)

print("카테고리 간 중심점 거리:")
for l1 in sorted(centroids.keys()):
    for l2 in sorted(centroids.keys()):
        if l1 < l2:
            dist = np.linalg.norm(centroids[l1] - centroids[l2])
            print(f"  {l1} ↔ {l2}: {dist:.4f}")

---
## 문제 3. ChromaDB CRUD + 메타데이터 필터링 시스템 (10점)

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer

TECH_DOCS = [
    {"id": "doc_001", "text": "쿠버네티스 파드(Pod)는 컨테이너의 최소 배포 단위입니다.", "category": "kubernetes", "difficulty": "beginner", "year": 2024},
    {"id": "doc_002", "text": "도커 이미지는 Dockerfile로부터 빌드됩니다.", "category": "docker", "difficulty": "beginner", "year": 2024},
    {"id": "doc_003", "text": "Redis는 인메모리 데이터 저장소로 캐싱에 사용됩니다.", "category": "database", "difficulty": "intermediate", "year": 2023},
    {"id": "doc_004", "text": "PostgreSQL의 EXPLAIN ANALYZE는 쿼리 실행 계획을 보여줍니다.", "category": "database", "difficulty": "advanced", "year": 2023},
    {"id": "doc_005", "text": "Kafka는 분산 이벤트 스트리밍 플랫폼입니다.", "category": "streaming", "difficulty": "intermediate", "year": 2024},
    {"id": "doc_006", "text": "Helm 차트는 쿠버네티스 애플리케이션 패키징 도구입니다.", "category": "kubernetes", "difficulty": "intermediate", "year": 2024},
    {"id": "doc_007", "text": "Prometheus는 메트릭 수집 및 모니터링 시스템입니다.", "category": "monitoring", "difficulty": "intermediate", "year": 2023},
    {"id": "doc_008", "text": "Terraform은 인프라를 코드로 관리하는 IaC 도구입니다.", "category": "iac", "difficulty": "intermediate", "year": 2024},
    {"id": "doc_009", "text": "gRPC는 고성능 RPC 프레임워크로 마이크로서비스에 적합합니다.", "category": "networking", "difficulty": "advanced", "year": 2023},
    {"id": "doc_010", "text": "GitHub Actions는 CI/CD 파이프라인 자동화 도구입니다.", "category": "cicd", "difficulty": "beginner", "year": 2024},
    {"id": "doc_011", "text": "HPA(Horizontal Pod Autoscaler)는 CPU 사용률 기반으로 파드를 자동 스케일링합니다.", "category": "kubernetes", "difficulty": "advanced", "year": 2024},
    {"id": "doc_012", "text": "ArgoCD는 GitOps 기반 쿠버네티스 배포 도구입니다.", "category": "cicd", "difficulty": "advanced", "year": 2024},
]

model = SentenceTransformer('all-MiniLM-L6-v2')

client = chromadb.Client()

collection = client.create_collection(name="tech_docs", metadata={"hnsw:space": "cosine"})

ids = [d["id"] for d in TECH_DOCS]
documents = [d["text"] for d in TECH_DOCS]
embeddings = model.encode(documents).tolist()
metadatas = [{"category": d["category"], "difficulty": d["difficulty"], "year": d["year"]} for d in TECH_DOCS]

collection.add(ids=ids, documents=documents, embeddings=embeddings, metadatas=metadatas)

print(f"적재된 문서 수: {collection.count()}")

# 검색 1: 기본 시맨틱 검색
query1 = "컨테이너 오케스트레이션 도구"
q1_emb = model.encode([query1]).tolist()
results1 = collection.query(query_embeddings=q1_emb, n_results=3)
print(f"\n[검색 1] '{query1}'")
for i, (doc, score) in enumerate(zip(results1['documents'][0], results1['distances'][0])):
    print(f"  {i+1}. (거리: {score:.4f}) {doc[:60]}...")

# 검색 2: 메타데이터 필터 (category == "kubernetes" AND difficulty != "beginner")
results2 = collection.query(
    query_embeddings=q1_emb,
    n_results=5,
    where={"$and": [{"category": {"$eq": "kubernetes"}}, {"difficulty": {"$ne": "beginner"}}]}
)
print(f"\n[검색 2] kubernetes + intermediate/advanced:")
for doc, meta in zip(results2['documents'][0], results2['metadatas'][0]):
    print(f"  [{meta['difficulty']}] {doc[:60]}...")

# 검색 3: 복합 필터 (year >= 2024 AND difficulty in ["beginner", "intermediate"])
results3 = collection.query(
    query_embeddings=q1_emb,
    n_results=5,
    where={"$and": [{"year": {"$gte": 2024}}, {"difficulty": {"$in": ["beginner", "intermediate"]}}]}
)
print(f"\n[검색 3] 2024년 + 초급/중급:")
for doc, meta in zip(results3['documents'][0], results3['metadatas'][0]):
    print(f"  [{meta['category']}/{meta['difficulty']}] {doc[:60]}...")

# 업데이트 & 삭제
collection.update(ids=["doc_003"], metadatas=[{"category": "database", "difficulty": "advanced", "year": 2023}])
collection.delete(ids=["doc_009"])
print(f"\n최종 문서 수: {collection.count()}")

---
## 문제 4. ETL 파이프라인 구축: 텍스트 → 청킹 → 임베딩 → ChromaDB (10점)

In [ ]:
RAW_DOCUMENTS = [
    {
        "title": "쿠버네티스 입문 가이드",
        "content": """쿠버네티스(Kubernetes, K8s)는 컨테이너화된 워크로드와 서비스를 관리하기 위한 이식 가능하고 확장 가능한 오픈소스 플랫폼입니다. 쿠버네티스는 선언적 구성과 자동화를 모두 지원합니다. 파드(Pod)는 쿠버네티스에서 생성하고 관리할 수 있는 배포 가능한 가장 작은 컴퓨팅 단위입니다. 파드는 하나 이상의 컨테이너 그룹으로, 스토리지와 네트워크를 공유하며 컨테이너를 실행하는 방법에 대한 명세를 갖습니다. 디플로이먼트(Deployment)는 파드와 레플리카셋에 대한 선언적 업데이트를 제공합니다. 서비스(Service)는 파드 집합에서 실행되는 애플리케이션을 네트워크 서비스로 노출하는 추상화된 방법입니다. 인그레스(Ingress)는 클러스터 외부에서 내부 서비스로의 HTTP/HTTPS 라우팅 규칙을 관리합니다.""",
        "source": "k8s-docs",
    },
    {
        "title": "FastAPI 프레임워크 소개",
        "content": """FastAPI는 파이썬 3.7+를 기반으로 하는 현대적이고 빠른 웹 프레임워크입니다. 주요 특징은 자동 API 문서화(Swagger UI, ReDoc), 타입 힌트 기반의 데이터 검증, 비동기(async/await) 지원입니다. Pydantic 모델을 사용하여 요청과 응답의 스키마를 정의하면 자동으로 유효성 검사가 수행됩니다. 미들웨어를 통해 CORS, 인증, 레이트 리미팅 등을 구현할 수 있습니다. 의존성 주입(Dependency Injection) 시스템으로 코드의 재사용성과 테스트 용이성을 높입니다. 배포는 uvicorn이나 gunicorn과 함께 사용하며, Docker 컨테이너로 패키징하는 것이 일반적입니다.""",
        "source": "fastapi-docs",
    },
    {
        "title": "Redis 캐싱 전략",
        "content": """Redis는 인메모리 데이터 구조 저장소로, 캐시, 메시지 브로커, 세션 저장소 등으로 활용됩니다. 주요 캐싱 패턴으로는 Cache-Aside(Lazy Loading), Write-Through, Write-Behind가 있습니다. Cache-Aside는 애플리케이션이 먼저 캐시를 조회하고, 없으면 DB에서 읽어온 후 캐시에 저장하는 방식입니다. TTL(Time To Live)을 설정하여 캐시 만료를 관리합니다. Redis의 데이터 구조로는 String, Hash, List, Set, Sorted Set, Stream 등이 있습니다. Redis Cluster를 사용하면 데이터를 여러 노드에 분산하여 고가용성과 확장성을 확보할 수 있습니다.""",
        "source": "redis-docs",
    },
]

def sliding_window_chunk(text, chunk_size=150, overlap=30):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

def extract_keywords(text, top_n=5):
    words = text.split()
    words = [w for w in words if len(w) >= 2]
    freq = {}
    for w in words:
        freq[w] = freq.get(w, 0) + 1
    sorted_words = sorted(freq.items(), key=lambda x: x[1], reverse=True)
    return [w for w, _ in sorted_words[:top_n]]

def build_chunk_metadata(chunk_text, doc_title, doc_source, chunk_index):
    keywords = extract_keywords(chunk_text)
    return {
        "title": doc_title,
        "source": doc_source,
        "chunk_index": chunk_index,
        "char_count": len(chunk_text),
        "keywords": ", ".join(keywords),
    }

def run_etl_pipeline(documents, collection_name="etl_docs"):
    client = chromadb.Client()
    model = SentenceTransformer('all-MiniLM-L6-v2')
    
    # 기존 컬렉션 삭제 후 재생성
    try:
        client.delete_collection(collection_name)
    except:
        pass
    collection = client.create_collection(name=collection_name)
    
    all_chunks = []
    all_ids = []
    all_metadatas = []
    
    for doc in documents:
        title = doc["title"]
        content = doc["content"]
        source = doc["source"]
        
        chunks = sliding_window_chunk(content)
        
        for i, chunk in enumerate(chunks):
            chunk_id = f"{source}_chunk_{i:03d}"
            metadata = build_chunk_metadata(chunk, title, source, i)
            
            all_chunks.append(chunk)
            all_ids.append(chunk_id)
            all_metadatas.append(metadata)
    
    embeddings = model.encode(all_chunks).tolist()
    collection.add(ids=all_ids, documents=all_chunks, embeddings=embeddings, metadatas=all_metadatas)
    
    return collection, len(all_chunks)

collection, total = run_etl_pipeline(RAW_DOCUMENTS)
print(f"✅ ETL 완료: {total}개 청크 적재")

test_queries = ["컨테이너 배포 방법", "캐시 만료 관리", "API 문서 자동화"]
for q in test_queries:
    results = collection.query(query_texts=[q], n_results=2)
    print(f"\n🔎 '{q}':")
    for doc, meta in zip(results['documents'][0], results['metadatas'][0]):
        print(f"  [{meta['source']}] {doc[:60]}...")

---
## 문제 5. BM25 + Vector 하이브리드 검색 엔진 (10점)

In [ ]:
from rank_bm25 import BM25Okapi

SEARCH_CORPUS = [
    "쿠버네티스 파드는 컨테이너의 최소 배포 단위입니다.",
    "도커 이미지는 Dockerfile로부터 레이어 단위로 빌드됩니다.",
    "Redis TTL을 설정하여 캐시 만료를 자동 관리합니다.",
    "FastAPI는 async/await 기반의 비동기 웹 프레임워크입니다.",
    "Kafka 토픽은 파티션으로 나뉘어 병렬 처리됩니다.",
    "PostgreSQL JSONB 타입은 JSON 데이터를 효율적으로 저장합니다.",
    "Prometheus AlertManager는 메트릭 임계값 초과 시 알림을 발송합니다.",
    "Terraform state 파일은 인프라의 현재 상태를 추적합니다.",
    "Nginx 리버스 프록시는 로드 밸런싱과 SSL 종료를 처리합니다.",
    "GitHub Actions workflow는 YAML 파일로 CI/CD를 정의합니다.",
    "gRPC는 Protocol Buffers를 사용한 고성능 RPC 프레임워크입니다.",
    "Elasticsearch는 역인덱스 기반의 풀텍스트 검색 엔진입니다.",
    "ArgoCD는 GitOps 방식으로 쿠버네티스 배포를 자동화합니다.",
    "Celery는 분산 태스크 큐로 비동기 작업 처리에 사용됩니다.",
    "Grafana 대시보드는 다양한 데이터소스의 메트릭을 시각화합니다.",
]

model = SentenceTransformer('all-MiniLM-L6-v2')
chroma_client = chromadb.Client()

tokenized_corpus = [doc.split() for doc in SEARCH_CORPUS]
bm25 = BM25Okapi(tokenized_corpus)

hybrid_collection = chroma_client.create_collection(name="hybrid_search", metadata={"hnsw:space": "cosine"})
corpus_embeddings = model.encode(SEARCH_CORPUS).tolist()
hybrid_collection.add(
    ids=[f"doc_{i}" for i in range(len(SEARCH_CORPUS))],
    documents=SEARCH_CORPUS,
    embeddings=corpus_embeddings,
)

def search_bm25(query, top_k=5):
    tokenized_query = query.split()
    scores = bm25.get_scores(tokenized_query)
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [(int(idx), float(scores[idx]), SEARCH_CORPUS[idx]) for idx in top_indices]

def search_vector(query, top_k=5):
    q_emb = model.encode([query]).tolist()
    results = hybrid_collection.query(query_embeddings=q_emb, n_results=top_k)
    out = []
    for doc_id, doc, dist in zip(results['ids'][0], results['documents'][0], results['distances'][0]):
        idx = int(doc_id.split('_')[1])
        out.append((idx, 1 - dist, doc))
    return out

def reciprocal_rank_fusion(bm25_results, vector_results, k=60):
    rrf_scores = {}
    doc_text = {}
    
    for rank, (idx, score, doc) in enumerate(bm25_results):
        rrf_scores[idx] = rrf_scores.get(idx, 0) + 1.0 / (k + rank + 1)
        doc_text[idx] = doc
    
    for rank, (idx, score, doc) in enumerate(vector_results):
        rrf_scores[idx] = rrf_scores.get(idx, 0) + 1.0 / (k + rank + 1)
        doc_text[idx] = doc
    
    sorted_results = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    return [(idx, score, doc_text[idx]) for idx, score in sorted_results]

def hybrid_search(query, top_k=5, mode="hybrid"):
    if mode == "bm25":
        return search_bm25(query, top_k)
    elif mode == "vector":
        return search_vector(query, top_k)
    elif mode == "hybrid":
        bm25_r = search_bm25(query, top_k * 2)
        vec_r = search_vector(query, top_k * 2)
        fused = reciprocal_rank_fusion(bm25_r, vec_r)
        return fused[:top_k]
    else:
        raise ValueError(f"Unknown mode: {mode}")

query = "컨테이너 배포 자동화"
for mode in ["bm25", "vector", "hybrid"]:
    results = hybrid_search(query, top_k=3, mode=mode)
    print(f"\n[{mode.upper()}] '{query}':")
    for idx, score, doc in results:
        print(f"  (score={score:.4f}) {doc[:60]}...")

---
## 문제 6. Hit Rate & MRR 평가 프레임워크 (10점)

In [ ]:
EVAL_DATASET = [
    ("쿠버네티스 배포 방법", [0, 12]),
    ("캐시 관리 전략", [2]),
    ("CI/CD 파이프라인 구성", [9, 12]),
    ("비동기 웹 서버 프레임워크", [3, 13]),
    ("메시지 큐 시스템", [4, 13]),
    ("데이터베이스 쿼리 최적화", [5]),
    ("인프라 모니터링 도구", [6, 14]),
    ("IaC 도구 비교", [7]),
    ("컨테이너 이미지 빌드", [1]),
    ("서비스 간 통신 프로토콜", [10, 8]),
]

def hit_rate_at_k(retrieved_indices, relevant_indices, k):
    top_k = retrieved_indices[:k]
    return 1.0 if any(idx in relevant_indices for idx in top_k) else 0.0

def mrr_at_k(retrieved_indices, relevant_indices, k):
    top_k = retrieved_indices[:k]
    for rank, idx in enumerate(top_k, 1):
        if idx in relevant_indices:
            return 1.0 / rank
    return 0.0

def evaluate_search_system(search_func, eval_dataset, ks=[1, 3, 5]):
    results = {}
    
    for k in ks:
        hit_rates = []
        mrrs = []
        
        for query, relevant_ids in eval_dataset:
            retrieved = search_func(query, k)
            
            hr = hit_rate_at_k(retrieved, relevant_ids, k)
            mr = mrr_at_k(retrieved, relevant_ids, k)
            
            hit_rates.append(hr)
            mrrs.append(mr)
        
        results[k] = {
            "hit_rate": np.mean(hit_rates),
            "mrr": np.mean(mrrs)
        }
    
    return results

print("=" * 60)
print(f"{'방식':>10} | {'K':>3} | {'Hit Rate':>10} | {'MRR':>10}")
print("-" * 60)

for mode in ["bm25", "vector", "hybrid"]:
    search_fn = lambda q, k, m=mode: [idx for idx, _, _ in hybrid_search(q, k, m)]
    eval_result = evaluate_search_system(search_fn, EVAL_DATASET)
    for k, metrics in eval_result.items():
        print(f"{mode:>10} | {k:>3} | {metrics['hit_rate']:>10.4f} | {metrics['mrr']:>10.4f}")

---
## 문제 7. FastAPI 벡터 검색 API 서버 + 캐시 + Rate Limiting (10점)

In [ ]:
from fastapi import FastAPI, HTTPException, Request
from fastapi.responses import JSONResponse
from pydantic import BaseModel, Field
from typing import List, Optional, Dict, Any
import uvicorn
import threading
import time
import hashlib
from collections import defaultdict

class SearchRequest(BaseModel):
    query: str
    top_k: int = 5
    mode: str = "hybrid"
    filters: Optional[Dict] = None

class SearchResult(BaseModel):
    id: str
    text: str
    score: float
    metadata: Dict[str, Any] = {}

class SearchResponse(BaseModel):
    query: str
    results: List[SearchResult]
    latency_ms: float
    cached: bool

class InMemoryCache:
    def __init__(self, ttl_seconds=60):
        self.cache = {}
        self.ttl = ttl_seconds
        self._hit_count = 0
    
    def get(self, key):
        if key in self.cache:
            ts, value = self.cache[key]
            if time.time() - ts < self.ttl:
                self._hit_count += 1
                return value
            else:
                del self.cache[key]
        return None
    
    def set(self, key, value):
        self.cache[key] = (time.time(), value)
    
    def make_key(self, query, top_k, mode):
        raw = f"{query}:{top_k}:{mode}"
        return hashlib.md5(raw.encode()).hexdigest()
    
    @property
    def hit_count(self):
        return self._hit_count

class RateLimiter:
    def __init__(self, max_requests=30, window_seconds=60):
        self.max_requests = max_requests
        self.window = window_seconds
        self.requests = defaultdict(list)
    
    def is_allowed(self, client_ip):
        now = time.time()
        self.requests[client_ip] = [t for t in self.requests[client_ip] if now - t < self.window]
        if len(self.requests[client_ip]) >= self.max_requests:
            return False
        self.requests[client_ip].append(now)
        return True

app = FastAPI(title="Hybrid Search API")
cache = InMemoryCache(ttl_seconds=60)
limiter = RateLimiter(max_requests=30)

@app.middleware("http")
async def rate_limit_middleware(request: Request, call_next):
    client_ip = request.client.host if request.client else "unknown"
    if not limiter.is_allowed(client_ip):
        return JSONResponse(status_code=429, content={"detail": "Rate limit exceeded"})
    return await call_next(request)

@app.post("/search", response_model=SearchResponse)
async def search_endpoint(request: SearchRequest):
    start = time.time()
    cache_key = cache.make_key(request.query, request.top_k, request.mode)
    
    cached_result = cache.get(cache_key)
    if cached_result:
        cached_result["cached"] = True
        cached_result["latency_ms"] = (time.time() - start) * 1000
        return cached_result
    
    raw_results = hybrid_search(request.query, request.top_k, request.mode)
    
    search_results = [
        SearchResult(id=f"doc_{idx}", text=doc, score=score, metadata={})
        for idx, score, doc in raw_results
    ]
    
    latency = (time.time() - start) * 1000
    response_data = {
        "query": request.query,
        "results": [r.model_dump() for r in search_results],
        "latency_ms": latency,
        "cached": False,
    }
    cache.set(cache_key, response_data)
    
    return SearchResponse(query=request.query, results=search_results, latency_ms=latency, cached=False)

@app.get("/health")
async def health():
    return {
        "status": "healthy",
        "document_count": len(SEARCH_CORPUS),
        "cache_size": len(cache.cache),
        "cache_hits": cache.hit_count,
    }

def run():
    import asyncio
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="warning")
    server = uvicorn.Server(config)
    loop.run_until_complete(server.serve())

server_thread = threading.Thread(target=run, daemon=True)
server_thread.start()
time.sleep(3)

import httpx
resp = httpx.post("http://localhost:8000/search", json={"query": "컨테이너 배포", "top_k": 3})
print(f"Status: {resp.status_code}")
print(f"Results: {resp.json()}")

---
## 문제 8 (킬러). 자동 청킹 전략 최적화 시스템 (10점)

In [ ]:
LONG_DOCUMENT = """
마이크로서비스 아키텍처는 대규모 애플리케이션을 작은 독립적인 서비스로 분해하는 설계 패턴입니다. 각 서비스는 특정 비즈니스 기능을 담당하며, 독립적으로 개발, 배포, 확장할 수 있습니다.

서비스 간 통신은 주로 REST API 또는 gRPC를 사용합니다. 비동기 통신이 필요한 경우 Kafka나 RabbitMQ 같은 메시지 브로커를 활용합니다. 이벤트 드리븐 아키텍처를 적용하면 서비스 간 결합도를 낮출 수 있습니다.

컨테이너화는 마이크로서비스 배포의 핵심입니다. Docker로 각 서비스를 패키징하고, 쿠버네티스로 오케스트레이션합니다. 서비스 메시(Service Mesh)인 Istio를 사용하면 트래픽 관리, 보안, 관찰 가능성을 투명하게 제공할 수 있습니다.

데이터 관리에서는 각 서비스가 자체 데이터베이스를 소유하는 Database per Service 패턴을 권장합니다. 분산 트랜잭션이 필요한 경우 Saga 패턴을 사용합니다. CQRS(Command Query Responsibility Segregation) 패턴으로 읽기와 쓰기를 분리하면 성능을 최적화할 수 있습니다.

모니터링과 로깅은 분산 시스템에서 필수적입니다. ELK 스택(Elasticsearch, Logstash, Kibana)으로 중앙 집중식 로깅을 구현하고, Prometheus와 Grafana로 메트릭을 수집하고 시각화합니다. 분산 추적은 Jaeger나 Zipkin을 사용하여 서비스 간 요청 흐름을 추적합니다.

보안은 API 게이트웨이에서 인증/인가를 중앙 처리하고, 서비스 간 통신은 mTLS로 암호화합니다. OAuth2와 JWT를 활용한 토큰 기반 인증이 일반적입니다. 서비스별 네트워크 정책으로 최소 권한 원칙을 적용합니다.
"""

EVAL_QUERIES = [
    ("마이크로서비스 간 통신 방법은?", "서비스 간 통신은 주로 REST API 또는 gRPC를 사용"),
    ("컨테이너 오케스트레이션 도구는?", "쿠버네티스로 오케스트레이션"),
    ("분산 트랜잭션 처리 패턴은?", "Saga 패턴"),
    ("로깅 시스템 구성은?", "ELK 스택"),
    ("서비스 간 보안 통신은?", "mTLS로 암호화"),
]

def chunk_fixed_size(text, size=150, overlap=30):
    chunks = []
    start = 0
    text = text.strip()
    while start < len(text):
        end = start + size
        chunks.append(text[start:end])
        start += size - overlap
    return chunks

def chunk_by_sentence(text):
    sentences = [s.strip() for s in text.strip().split('.') if s.strip()]
    chunks = []
    group_size = 2
    for i in range(0, len(sentences), group_size):
        group = sentences[i:i+group_size]
        chunks.append('. '.join(group) + '.')
    return chunks

def chunk_semantic(text, model, threshold=0.5):
    sentences = [s.strip() for s in text.strip().split('.') if s.strip()]
    if len(sentences) <= 1:
        return [text.strip()]
    
    embeddings = model.encode(sentences)
    
    split_points = [0]
    for i in range(len(sentences) - 1):
        sim = np.dot(embeddings[i], embeddings[i+1]) / (np.linalg.norm(embeddings[i]) * np.linalg.norm(embeddings[i+1]))
        if sim < threshold:
            split_points.append(i + 1)
    split_points.append(len(sentences))
    
    chunks = []
    for i in range(len(split_points) - 1):
        chunk_sentences = sentences[split_points[i]:split_points[i+1]]
        chunks.append('. '.join(chunk_sentences) + '.')
    return chunks

def evaluate_chunking_strategy(chunks, eval_queries, collection_name):
    client = chromadb.Client()
    model = SentenceTransformer('all-MiniLM-L6-v2')
    
    try:
        client.delete_collection(collection_name)
    except:
        pass
    col = client.create_collection(name=collection_name)
    
    embs = model.encode(chunks).tolist()
    col.add(
        ids=[f"chunk_{i}" for i in range(len(chunks))],
        documents=chunks,
        embeddings=embs,
    )
    
    hits = 0
    for query, answer_keyword in eval_queries:
        results = col.query(query_texts=[query], n_results=3)
        for doc in results['documents'][0]:
            if answer_keyword in doc:
                hits += 1
                break
    
    return hits / len(eval_queries)

model = SentenceTransformer('all-MiniLM-L6-v2')
strategies = {
    "fixed_size": chunk_fixed_size(LONG_DOCUMENT),
    "sentence": chunk_by_sentence(LONG_DOCUMENT),
    "semantic": chunk_semantic(LONG_DOCUMENT, model, threshold=0.5),
}

print("=" * 50)
print(f"{'전략':>12} | {'청크 수':>7} | {'Hit Rate@3':>11}")
print("-" * 50)
best_name = None
best_hr = -1
for name, chunks in strategies.items():
    hr = evaluate_chunking_strategy(chunks, EVAL_QUERIES, f"eval_{name}")
    if hr > best_hr:
        best_hr = hr
        best_name = name
    print(f"{name:>12} | {len(chunks):>7} | {hr:>11.4f}")

print(f"\n🏆 최적 전략: {best_name}")

---
## 문제 9 (킬러). 비동기 배치 임베딩 + 동기 대비 성능 비교 (10점)

In [ ]:
import asyncio
import nest_asyncio
nest_asyncio.apply()

BATCH_DOCS = [f"테스트 문서 {i}: {'AI 기술' if i%3==0 else '클라우드 인프라' if i%3==1 else '데이터 분석'} 관련 내용입니다. 문서 번호는 {i}번입니다." for i in range(100)]

class BatchEmbeddingProcessor:
    def __init__(self, model_name='all-MiniLM-L6-v2', batch_size=10, max_retries=2):
        self.model = SentenceTransformer(model_name)
        self.batch_size = batch_size
        self.max_retries = max_retries
        self.processed = 0
        self.failed_batches = []
    
    def _split_batches(self, documents):
        return [documents[i:i+self.batch_size] for i in range(0, len(documents), self.batch_size)]
    
    def process_sync(self, documents):
        batches = self._split_batches(documents)
        all_embeddings = []
        start = time.time()
        self.processed = 0
        
        for i, batch in enumerate(batches):
            for attempt in range(self.max_retries + 1):
                try:
                    emb = self.model.encode(batch)
                    all_embeddings.append(emb)
                    self.processed += len(batch)
                    print(f"\r  [동기] 진행: {self.processed}/{len(documents)}", end="")
                    break
                except Exception as e:
                    if attempt == self.max_retries:
                        self.failed_batches.append(i)
        
        elapsed = time.time() - start
        print()
        return np.vstack(all_embeddings), elapsed
    
    async def _embed_batch_async(self, batch, batch_idx):
        loop = asyncio.get_event_loop()
        for attempt in range(self.max_retries + 1):
            try:
                embeddings = await loop.run_in_executor(None, self.model.encode, batch)
                self.processed += len(batch)
                return embeddings
            except Exception as e:
                if attempt == self.max_retries:
                    self.failed_batches.append(batch_idx)
                    return None
    
    async def process_async(self, documents):
        batches = self._split_batches(documents)
        start = time.time()
        self.processed = 0
        self.failed_batches = []
        
        tasks = [self._embed_batch_async(batch, i) for i, batch in enumerate(batches)]
        results = await asyncio.gather(*tasks)
        
        valid_results = [r for r in results if r is not None]
        
        elapsed = time.time() - start
        print(f"  [비동기] 진행: {self.processed}/{len(documents)}")
        all_embeddings = np.vstack(valid_results)
        return all_embeddings, elapsed

processor = BatchEmbeddingProcessor(batch_size=10)

sync_embeddings, sync_time = processor.process_sync(BATCH_DOCS)
print(f"[동기] {len(BATCH_DOCS)}개 문서, 소요시간: {sync_time:.2f}초")

async_embeddings, async_time = asyncio.get_event_loop().run_until_complete(
    processor.process_async(BATCH_DOCS)
)
print(f"[비동기] {len(BATCH_DOCS)}개 문서, 소요시간: {async_time:.2f}초")
print(f"\n속도 비교: 비동기가 {sync_time/async_time:.1f}배 {'빠름' if async_time < sync_time else '느림'}")
print(f"실패 배치: {len(processor.failed_batches)}개")

# 결과 일치 확인
cos_sims = [np.dot(sync_embeddings[i], async_embeddings[i]) / (np.linalg.norm(sync_embeddings[i]) * np.linalg.norm(async_embeddings[i])) for i in range(len(sync_embeddings))]
print(f"동기/비동기 결과 평균 코사인 유사도: {np.mean(cos_sims):.6f}")

---
## 문제 10 (킬러). Production RAG 평가 파이프라인 통합 (10점)

In [ ]:
from openai import OpenAI
import json as json_module

RAG_DOCUMENTS = [
    "쿠버네티스(K8s)는 컨테이너 오케스트레이션 플랫폼으로 자동 스케일링, 셀프 힐링, 롤링 업데이트를 지원합니다.",
    "Docker Compose는 멀티 컨테이너 앱을 YAML로 정의하고 단일 명령으로 실행하는 도구입니다.",
    "Terraform은 HashiCorp의 IaC 도구로 AWS, GCP, Azure 등 멀티클라우드 인프라를 코드로 관리합니다.",
    "Redis Sentinel은 마스터-슬레이브 구조에서 자동 장애 복구(failover)를 제공합니다.",
    "Kafka Connect는 데이터베이스, 파일시스템 등 외부 시스템과 Kafka 간의 데이터 이동을 자동화합니다.",
    "Prometheus의 PromQL은 시계열 데이터를 쿼리하는 함수형 언어로 rate, histogram_quantile 등의 함수를 제공합니다.",
    "Istio 서비스 메시는 사이드카 프록시(Envoy)를 통해 트래픽 관리, mTLS, 관찰 가능성을 투명하게 제공합니다.",
    "FastAPI의 Depends()는 의존성 주입 시스템으로 인증, DB 세션, 캐시 등을 엔드포인트에 자동 주입합니다.",
]

RAG_EVAL = [
    {"question": "쿠버네티스의 자동 복구 기능은 무엇인가요?", "reference": "쿠버네티스는 셀프 힐링 기능으로 장애가 발생한 파드를 자동으로 재시작합니다.", "relevant_docs": [0]},
    {"question": "여러 클라우드에 인프라를 배포하는 도구는?", "reference": "Terraform은 AWS, GCP, Azure 등 멀티클라우드 인프라를 코드로 관리하는 IaC 도구입니다.", "relevant_docs": [2]},
    {"question": "Kafka와 외부 시스템을 연결하는 방법은?", "reference": "Kafka Connect를 사용하여 데이터베이스 등 외부 시스템과의 데이터 이동을 자동화할 수 있습니다.", "relevant_docs": [4]},
    {"question": "서비스 간 보안 통신을 투명하게 처리하는 방법은?", "reference": "Istio 서비스 메시의 사이드카 프록시를 통해 mTLS 기반의 암호화된 통신을 투명하게 제공합니다.", "relevant_docs": [6]},
    {"question": "FastAPI에서 인증 로직을 재사용하는 방법은?", "reference": "FastAPI의 Depends() 의존성 주입 시스템을 통해 인증 로직을 엔드포인트에 자동 주입할 수 있습니다.", "relevant_docs": [7]},
]

class RAGEvaluationPipeline:
    def __init__(self, documents):
        self.documents = documents
        self.model = SentenceTransformer('all-MiniLM-L6-v2')
        self.openai_client = OpenAI()
        
        self.client = chromadb.Client()
        try:
            self.client.delete_collection("rag_eval")
        except:
            pass
        self.collection = self.client.create_collection(name="rag_eval")
        
        embs = self.model.encode(documents).tolist()
        self.collection.add(
            ids=[f"doc_{i}" for i in range(len(documents))],
            documents=documents,
            embeddings=embs,
        )
    
    def retrieve(self, question, top_k=3):
        results = self.collection.query(query_texts=[question], n_results=top_k)
        docs = results['documents'][0]
        indices = [int(id_.split('_')[1]) for id_ in results['ids'][0]]
        return docs, indices
    
    def generate_answer(self, question, context_docs):
        context = "\n".join([f"[{i+1}] {doc}" for i, doc in enumerate(context_docs)])
        prompt = f"""다음 참고 문서만을 기반으로 질문에 답하세요. 참고 문서에 없는 내용은 답하지 마세요.

[참고 문서]
{context}

[질문] {question}

[답변]"""
        response = self.openai_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            max_tokens=200,
        )
        return response.choices[0].message.content.strip()
    
    def evaluate_retrieval(self, eval_data, top_k=3):
        mrrs = []
        for item in eval_data:
            _, retrieved_indices = self.retrieve(item["question"], top_k)
            relevant = item["relevant_docs"]
            rr = 0.0
            for rank, idx in enumerate(retrieved_indices, 1):
                if idx in relevant:
                    rr = 1.0 / rank
                    break
            mrrs.append(rr)
        return np.mean(mrrs)
    
    def evaluate_answer_rouge(self, generated, reference):
        # ROUGE-L 직접 구현 (LCS 기반)
        gen_tokens = generated.split()
        ref_tokens = reference.split()
        
        if not gen_tokens or not ref_tokens:
            return 0.0
        
        # LCS 길이 계산
        m, n = len(ref_tokens), len(gen_tokens)
        dp = [[0] * (n + 1) for _ in range(m + 1)]
        for i in range(1, m + 1):
            for j in range(1, n + 1):
                if ref_tokens[i-1] == gen_tokens[j-1]:
                    dp[i][j] = dp[i-1][j-1] + 1
                else:
                    dp[i][j] = max(dp[i-1][j], dp[i][j-1])
        lcs_len = dp[m][n]
        
        precision = lcs_len / n if n > 0 else 0
        recall = lcs_len / m if m > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        return f1
    
    def evaluate_faithfulness(self, question, answer, context_docs):
        prompt = f"""다음 답변이 제공된 참고 문서에만 근거하는지 평가하세요.

[참고 문서]
{chr(10).join(context_docs)}

[답변]
{answer}

답변의 모든 주장이 참고 문서에 근거하면 faithful=true, 문서에 없는 정보가 포함되면 faithful=false입니다.
JSON으로만 응답하세요:
{{"faithful": true/false, "score": 0.0~1.0, "reason": "한줄 설명"}}"""
        response = self.openai_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            max_tokens=200,
        )
        try:
            text = response.choices[0].message.content.strip()
            # JSON 부분만 추출
            if '```' in text:
                text = text.split('```')[1].replace('json', '').strip()
            return json_module.loads(text)
        except:
            return {"faithful": False, "score": 0.0, "reason": "파싱 실패"}
    
    def run_full_evaluation(self, eval_data):
        results = []
        
        for item in eval_data:
            q = item["question"]
            ref = item["reference"]
            
            retrieved_docs, _ = self.retrieve(q)
            answer = self.generate_answer(q, retrieved_docs)
            rouge = self.evaluate_answer_rouge(answer, ref)
            faith = self.evaluate_faithfulness(q, answer, retrieved_docs)
            
            results.append({
                "question": q,
                "answer": answer,
                "rouge_l": rouge,
                "faithful": faith["faithful"] if faith else None,
                "faith_score": faith["score"] if faith else None,
            })
        
        mrr = self.evaluate_retrieval(eval_data)
        
        print("=" * 65)
        print("RAG 평가 종합 리포트")
        print("=" * 65)
        print(f"  검색 품질 (MRR@3): {mrr:.4f}")
        print(f"  평균 ROUGE-L: {np.mean([r['rouge_l'] for r in results]):.4f}")
        faithful_count = sum(1 for r in results if r['faithful'])
        print(f"  Faithfulness 비율: {faithful_count/len(results):.0%}")
        faith_scores = [r['faith_score'] for r in results if r['faith_score'] is not None]
        print(f"  평균 Faith Score: {np.mean(faith_scores):.4f}")
        print("-" * 65)
        for r in results:
            print(f"  Q: {r['question'][:40]}...")
            print(f"    ROUGE: {r['rouge_l']:.4f} | Faithful: {r['faithful']} ({r['faith_score']:.2f})")
        
        return results

pipeline = RAGEvaluationPipeline(RAG_DOCUMENTS)
results = pipeline.run_full_evaluation(RAG_EVAL)